# Lesson 2: Encapsulation and Abstraction

## Learning Objectives

By the end of this lesson, you will be able to:
- Explain the principle of Encapsulation and its importance.
- Use naming conventions (`_` and `__`) to signal private attributes.
- Implement properties with `@property` and `@setter` decorators to control attribute access.
- Explain the principle of Abstraction and how it simplifies complex systems.
- Design classes that expose a clean, public interface while hiding implementation details.

## Why This Topic Matters

In the last lesson, we created a `SimpleDataset` class. It worked, but it had a major flaw: its internal data was completely exposed. Anyone could accidentally (or intentionally) corrupt the object's state, like setting `iris_dataset.num_rows = -100`, which makes no sense!

**Encapsulation** is about building protective walls around our data. It means bundling the data (attributes) and the methods that operate on that data within one unit (the class) and controlling access to that data. This prevents unintended modifications and makes our code more robust and predictable.

**Abstraction** is about hiding complexity. Think about driving a car. You only need to know how to use the steering wheel, pedals, and gear stick (the public interface). You don't need to know the complex engineering details of the engine or transmission (the implementation). Abstraction in OOP applies the same idea: we create simple interfaces for complex logic, making our classes easier to use.

## Intuition-First Explanation

**Encapsulation is like a bank vault.**
- The money inside is the **data** (attributes).
- The thick steel walls are the **encapsulation** itself.
- You can't just walk in and grab the money. You have to go through a bank teller (the **public methods/interface**). The teller has specific rules they must follow to give you money or accept a deposit. They provide controlled access.

**Abstraction is like a TV remote control.**
- It has a few simple buttons: power, volume up, volume down, channel change.
- When you press 'volume up', a lot happens behind the scenes: an infrared signal is sent, the TV's receiver decodes it, and the audio processor adjusts the amplifier's output.
- You don't care about any of that complexity. You just want the simple abstraction of 'volume up'. The remote is the **public interface** that hides the complex **implementation**.

## Core Concept: Encapsulation in Python

Python doesn't have true 'private' variables like other languages (e.g., Java). Instead, it relies on naming conventions to signal intent.

### 1. Protected Attributes: Single Underscore (`_`)

A single underscore prefix (e.g., `_my_attribute`) is a convention that tells other developers: "This is an internal attribute. You shouldn't touch this directly unless you know what you're doing. It's not part of the public interface of this class."

It's a 'gentleman's agreement'. The interpreter doesn't stop you from accessing it, but it's a strong hint to stay away.

In [ ]:
class BankAccount:
    def __init__(self, owner, starting_balance):
        self.owner = owner # Public attribute
        self._balance = starting_balance # "Protected" attribute

    def deposit(self, amount):
        if amount > 0:
            self._balance += amount
            print(f"Deposited ${amount}. New balance: ${self._balance}")
        else:
            print("Deposit amount must be positive.")

    def withdraw(self, amount):
        if 0 < amount <= self._balance:
            self._balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self._balance}")
        else:
            print("Invalid withdrawal amount.")

    def get_balance(self):
        """This is the 'public' way to access the balance."""
        return self._balance

# --- Usage ---
my_account = BankAccount("John Doe", 1000)

# Good practice: Use the methods
my_account.deposit(500)
my_account.withdraw(200)
print(f"Current balance via method: ${my_account.get_balance()}")

# Bad practice: Accessing the protected attribute directly
# This is possible but breaks encapsulation. You are bypassing the rules.
my_account._balance = -9999
print(f"Current balance after direct access: ${my_account._balance}")

By setting the balance to `-9999`, we've put the object in an invalid state. The `withdraw` and `deposit` methods act as the 'bank tellers' that protect the data, but we just walked straight into the vault.

### 2. Name Mangling: Double Underscore (`__`)

A double underscore prefix (e.g., `__my_attribute`) triggers a feature called **name mangling**. Python automatically changes the name of the attribute to `_ClassName__attribute_name`.

This makes it much harder to access the attribute directly from outside the class, especially in the context of inheritance (which we'll cover later). It's a stronger signal than the single underscore.

In [ ]:
class BankVault:
    def __init__(self, secret_code):
        self.__secret = secret_code # Name-mangled attribute

vault = BankVault("1234")

try:
    # This will fail!
    print(vault.__secret)
except AttributeError as e:
    print(f"Error: {e}")

# You can still access it if you know the mangled name, but you shouldn't.
print("Accessing via mangled name:")
print(vault._BankVault__secret)

## A More Pythonic Way: Properties

Writing `get_balance()` and `set_balance()` methods is common in other languages, but Python offers a cleaner, more elegant solution: **properties**. 

Properties allow you to expose what *looks like* a public attribute, but behind the scenes, it's actually running your getter and setter methods. This gives you the best of both worlds: a clean interface and controlled access.

We use decorators (`@`) to define properties.

In [ ]:
class SmartBankAccount:
    def __init__(self, owner, starting_balance):
        self.owner = owner
        # The actual data is stored in a 'private' attribute
        self.__balance = starting_balance 

    # This is the 'getter' property.
    # It allows us to access 'account.balance' as if it were a public attribute.
    @property
    def balance(self):
        print("(Getting balance...)")
        return self.__balance

    # This is the 'setter' property.
    # It runs when we try to assign a value, e.g., 'account.balance = 1500'.
    @balance.setter
    def balance(self, new_balance):
        print("(Setting balance...)")
        if new_balance < 0:
            print("Cannot set a negative balance. Transaction cancelled.")
            return # Stop the execution
        self.__balance = new_balance
        print("Balance set successfully.")

# --- Usage ---
account = SmartBankAccount("Jane Doe", 1000)

# Accessing the 'balance' property. This calls the getter method.
print(f"Initial balance: ${account.balance}")

print("\n--- Trying to set a valid new balance ---")
# Assigning to the 'balance' property. This calls the setter method.
account.balance = 1500
print(f"New balance: ${account.balance}")

print("\n--- Trying to set an invalid new balance ---")
# The setter's logic protects our data!
account.balance = -500
print(f"Final balance: ${account.balance}")

Notice how the interface is clean (`account.balance`), but our validation logic runs automatically. This is the power of encapsulation with properties.

## Core Concept: Abstraction in Practice

Abstraction is less about a specific syntax and more about a design philosophy. It's about deciding what the user of your class *needs* to know and what they *don't*. 

Let's refine our `Dataset` class from Lesson 1 with these principles in mind.

In [ ]:
import pandas as pd
from io import StringIO

class DataProcessor:
    """A class that abstracts the process of loading and cleaning a dataset."""

    def __init__(self, data_string):
        # Implementation detail: The raw, unprocessed data.
        # We use a single underscore to signal it's for internal use.
        self._raw_data_string = data_string
        
        # Implementation detail: The processed DataFrame.
        # It starts as None and is only created when needed.
        self.__dataframe = None
        print("DataProcessor initialized.")

    # --- Private Helper Method ---
    def _load_and_clean(self):
        """An internal method to handle the complex loading and cleaning logic."""
        print("Internal: Loading and cleaning data...")
        # Pretend there's complex logic here
        temp_df = pd.read_csv(StringIO(self._raw_data_string))
        temp_df.columns = temp_df.columns.str.strip().str.lower().str.replace(' ', '_')
        # Let's say we always drop rows with any missing values in this processor
        temp_df.dropna(inplace=True)
        self.__dataframe = temp_df
        print("Internal: Data loaded and cleaned.")

    # --- Public Interface ---
    @property
    def data(self):
        """The main public property to get the cleaned data.
           It abstracts away the loading process (lazy loading).
        """
        print("Public: Accessing 'data' property...")
        if self.__dataframe is None:
            # The data is only loaded the first time it's requested!
            self._load_and_clean()
        return self.__dataframe

    def get_summary(self):
        """A public method that provides a high-level summary."""
        print("Public: Generating summary...")
        # This method uses the public 'data' property, ensuring data is loaded.
        return self.data.describe()

### Using the Abstracted Class

The user of this class has a very simple experience. They don't need to know about `_raw_data_string`, `__dataframe`, or `_load_and_clean`. They just use the clean public interface.

In [ ]:
messy_csv_data = "  Sepal Length , Sepal Width, Petal Length, Petal Width, Species\n5.1,3.5,1.4,0.2,setosa\n,,1.4,0.2,setosa\n7.0,3.2,,1.4,versicolor"

# 1. Create the object. Notice no data is loaded yet.
processor = DataProcessor(messy_csv_data)

print("\n----------------------------------------\n")

# 2. The first time we access '.data', the internal loading/cleaning is triggered.
print("First time accessing data:")
cleaned_data = processor.data
print(cleaned_data)

print("\n----------------------------------------\n")

# 3. The second time, the data is already loaded and is returned instantly.
print("Second time accessing data:")
more_data = processor.data
print(more_data)

print("\n----------------------------------------\n")

# 4. The user can just call the high-level method without worrying about the state.
print("Getting summary:")
summary = processor.get_summary()

This is a perfect example of abstraction. The complex, messy details (`_load_and_clean`) are hidden, and a simple, clean interface (`.data`, `.get_summary()`) is exposed to the user.

## Recap

- **Encapsulation**: Protecting object data by bundling it with methods and controlling access. Think 'bank vault'.
- **Abstraction**: Hiding complex implementation details behind a simple public interface. Think 'TV remote'.
- **Single Underscore (`_`)**: A convention for 'protected' internal attributes. A hint to not use them directly.
- **Double Underscore (`__`)**: Triggers 'name mangling' (`_ClassName__attribute`), making direct access much harder.
- **`@property`**: A 'getter' that lets you run code when an attribute is accessed. It makes a method look like an attribute.
- **`@*.setter`**: A 'setter' that lets you run code (like validation) when a value is assigned to an attribute.
- Good class design involves thinking about what should be **public** and what should be **private**.

## Suggested Next Lesson

We've learned how to build a single, robust class. But the real power of OOP comes from how classes can relate to each other. In the next lesson, we'll explore **Inheritance**, a mechanism that allows you to create a new class that is a specialized version of an existing class, reusing its code and extending its functionality.